<a href="https://colab.research.google.com/github/viktoriagajdosova/Enhancing-Meta-Research-in-Psychology-by-Generative-AI/blob/main/Enhancing-Meta-Research-in-Psychology-by-Generative-AI/pipelines/01_embedding-for-psychometrics/content_overlap_embedding_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 1: Installation, Imports, and GPU Setup (UNIVERSAL VERSION)

# 1. Install necessary libraries
!pip install openai google-genai numpy scikit-learn sentence-transformers torch --quiet

# 2. Imports
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import torch
import random
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import userdata

# Imports for Models
from openai import OpenAI
from google import genai
from sentence_transformers import SentenceTransformer

#  Determinism
def set_all_seeds(seed_value=42):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        print(f"GPU Determinism set for True and seed {seed_value}.")
    else:
        print(f"Seed {seed_value} set for CPU.")

# 3. Set device to GPU (cuda)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
set_all_seeds(42)


print("---")
print(f"Core libraries installed and imported.")
print(f"Using device: {device} (For local models)")
print("---")

Seed 42 set for CPU.
---
Core libraries installed and imported.
Using device: cpu (For local models)
---


In [2]:
# CELL 2: API/Model Initialization and Universal Embedding Function

# Essential imports
from google.colab import userdata
from openai import OpenAI
from google import genai
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity

# ⚠️ SWITCH: SELECT ONE PLATFORM
# Choose 'OPENAI', 'GEMINI', 'E5_LOCAL'

API_PLATFORM = 'OPENAI'  #

# API_PLATFORM = 'GEMINI'
# API_PLATFORM = 'OPENAI'
# API_PLATFORM = 'E5_LOCAL'

# --- CLIENTS ---
client = None
MODEL_NAME = None
OUTPUT_DIMENSIONALITY = None
API_KEY = None

# --- MODEL CONFIGURATION ---
if API_PLATFORM == 'OPENAI':
    # 1. OpenAI Configuration
    API_KEY = userdata.get('OPENAI_API_KEY')
    MODEL_NAME = 'text-embedding-3-large'
    OUTPUT_DIMENSIONALITY = 3072
    client = OpenAI(api_key=API_KEY)

elif API_PLATFORM == 'GEMINI':
    # 2. Gemini Configuration
    API_KEY = userdata.get('GOOGLE_API_KEY2')
    MODEL_NAME = 'gemini-embedding-001'
    try:
        client = genai.Client(api_key=API_KEY)
    except Exception as e:
        print(f"FATAL ERROR: Gemini client initialization failed: {e}")
        client = None

elif API_PLATFORM == 'E5_LOCAL':
    # 3. E5 Local Configuration
    API_KEY = None
    MODEL_NAME = 'intfloat/e5-large-v2'
    OUTPUT_DIMENSIONALITY = 1024

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    try:
        client = SentenceTransformer(MODEL_NAME, device=device)
        client.eval()
        print(f"E5 model '{MODEL_NAME}' successfully loaded on device: {device}")
    except Exception as e:
        print(f"FATAL ERROR: E5 model initialization failed: {e}")
        client = None

else:
    raise ValueError("Invalid API_PLATFORM value.")

# --- CHECK AND UNIVERSAL FUNCTION ---

if client is None:
     print("🔴 Error: API Client/Model is not initialized.")
# Added GTE_LOCAL to the list of models that do not require an API Key
elif API_PLATFORM not in ['E5_LOCAL', 'DWULFF_LOCAL', 'GTE_LOCAL'] and not API_KEY:
    print(f"ERROR: API key for {API_PLATFORM} was not found.")

print(f"--- ACTIVE CONFIGURATION ---")
print(f"Platform: {API_PLATFORM}")
print(f"Model: {MODEL_NAME}")
print(f"Dimension: {OUTPUT_DIMENSIONALITY if OUTPUT_DIMENSIONALITY else 'Default'}")
print("----------------------------")


def get_embeddings_for_texts(texts: list) -> np.ndarray:
    """Generates embeddings for a list of texts using the selected API/Model."""
    if client is None:
         print("🔴 Error: API Client/Model is not initialized.")
         return np.array([])

    if API_PLATFORM == 'OPENAI':
        # OpenAI SDK call
        response = client.embeddings.create(
            input=texts,
            model=MODEL_NAME,
            dimensions=OUTPUT_DIMENSIONALITY
        )
        embeddings_list = [item.embedding for item in response.data]
        return np.array(embeddings_list)

    elif API_PLATFORM == 'GEMINI':
        try:
            response = client.models.embed_content(
                model=MODEL_NAME,
                contents=texts,
            )
            embeddings_list = [result.values for result in response.embeddings]
            return np.array(embeddings_list)

        except Exception as e:
            print(f"🔴 Error calling Gemini API: {e}")
            return np.array([])

    elif API_PLATFORM == 'E5_LOCAL':
        # E5 LOGIC: Requires 'query: ' prefix
        try:
            prefixed_texts = [f"query: {t}" for t in texts]
            embeddings_matrix = client.encode(
                prefixed_texts,
                convert_to_numpy=True,
                show_progress_bar=False,
                device=client.device
            )
            return embeddings_matrix

        except Exception as e:
            print(f"🔴 Error calling E5/SentenceTransformer: {e}")
            return np.array([])

    return np.array([])

--- ACTIVE CONFIGURATION ---
Platform: OPENAI
Model: text-embedding-3-large
Dimension: 3072
----------------------------


In [3]:
# CELL 3: Questionnaire Similarity (Directional Coverage & Symmetrical Overlap)

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

print("--- 1. INITIALIZATION CHECK ---")
if 'get_embeddings_for_texts' not in locals() and 'get_embeddings_for_texts' not in globals():
    raise RuntimeError("⚠️ FATAL ERROR: Function 'get_embeddings_for_texts' not found. Please run CELL 2 first.")

# -----------------------------------------------------------------
# STEP A: DATA DEFINITION
# -----------------------------------------------------------------
print("\n--- STEP A: DATA DEFINITION ---")

# QUESTIONNAIRE A:
list_A_items = [
"Has anyone ever tried to take something directly from you by using force or the threat of force, such as a stick-up or mugging?",
"Has anyone ever attempted to rob you or actually robbed you (i.e., stolen your personal belongings)?",
"Has anyone ever attempted to or succeeded in breaking into your home when you were not there?",
"Has anyone ever attempted to or succeed in breaking into your home while you were there?",
"Have you ever had a serious accident at work, in a car, or somewhere else?",
"Have you ever experienced a natural disaster such as a tornado, hurricane, flood or major earthquake, etc., where you felt you or your loved ones were in danger of death or injury?",
"Have you ever experienced a ‘‘man-made’’ disaster such as a train crash, building collapse, bank robbery, fire, etc., where you felt you or your loved ones were in danger of death or injury?",
"Have you ever been exposed to dangerous chemicals or radioactivity that might threaten your health?",
"Have you ever been in any other situation in which you were seriously injured?",
"Have you ever been in any other situation in which you feared you might be killed or seriously injured?",
"Have you ever seen someone seriously injured or killed?",
"Have you ever seen dead bodies (other than at a funeral) or had to handle dead bodies for any reason?",
"Have you ever had a close friend or family member murdered, or killed by a drunk driver?",
"Have you ever had a spouse, romantic partner, or child die?",
"Have you ever had a serious or life-threatening illness? ",
"Have you ever received news of a serious injury, life-threatening illness, or unexpected death of someone close to you?",
"Have you ever had to engage in combat while in military service in an official or unofficial war zone?",
"Has anyone ever made you have intercourse or oral or anal sex against your will?",
"Has anyone ever touched private parts of your body, or made you touch theirs, under force or threat?",
"Other than incidents mentioned in Questions 18 and 19, have there been any other situations in which another person tried to force you to have an unwanted sexual contact?",
"Has anyone, including family members or friends, ever attacked you with a gun, knife, or some other weapon?",
"Has anyone, including family members or friends, ever attacked you without a weapon and seriously injured you?",
"Has anyone in your family ever beaten, spanked, or pushed you hard enough to cause injury?",
"Have you experienced any other extraordinarily stressful situation or event that is not covered above?"
]

# QUESTIONNAIRE B:
list_B_items = [
"Natural disasters",
"Motor vehicle accidents",
"Other accidents",
"Warfare or combat",
"Sudden death of close friend or loved one",
"Robbery involving a weapon",
"Severe assault by acquaintance or stranger",
"Witness to severe assault of acquaintance or stranger",
"Threat of death or serious bodily harm",
"Childhood physical abuse",
"Witness to family violence",
"Physical abuse by an intimate partner",
"Sexual abuse before age 13 by someone at least 5 year older",
"Sexual abuse before age 13 by someone close in age",
"Sexual abuse during adolescence",
"Sexual abuse as an adult",
"Stalking",
"Life-threatening illness",
"Life-threatening or permanently disabling event for loved one",
"Miscarriage",
"Abortion",
]

# Text cleaning
list_A_items = [t.strip() for t in list_A_items if t.strip()]
list_B_items = [t.strip() for t in list_B_items if t.strip()]

print(f"✅ Questionnaire A loaded: {len(list_A_items)} items.")
print(f"✅ Questionnaire B loaded: {len(list_B_items)} items.")


# -----------------------------------------------------------------
# STEP B: EMBEDDING GENERATION
# -----------------------------------------------------------------
print("\n--- STEP B: EMBEDDING GENERATION ---")

try:
    embeddings_A = get_embeddings_for_texts(list_A_items)
    embeddings_B = get_embeddings_for_texts(list_B_items)
    print(f"✅ Vectors generated successfully.")

except Exception as e:
    print(f"❌ ERROR during embedding generation: {e}")
    embeddings_A = None
    embeddings_B = None


# -----------------------------------------------------------------
# STEP C: SIMILARITY CALCULATIONS
# -----------------------------------------------------------------
print("\n--- STEP C: CALCULATIONS ---")

if embeddings_A is not None and embeddings_B is not None:

    # --- METHOD: PAIRWISE MATRIX (Best-match mapping) ---
    # Calculates cosine similarity between every item in A and every item in B
    similarity_matrix = cosine_similarity(embeddings_A, embeddings_B)

    # How well does Questionnaire A cover the content of B?
    max_scores_for_B = similarity_matrix.max(axis=0)
    coverage_B_by_A = max_scores_for_B.mean()

    # How well does Questionnaire B cover the content of A?
    max_scores_for_A = similarity_matrix.max(axis=1)
    coverage_A_by_B = max_scores_for_A.mean()

    # --- METHOD: UNIFIED SCORE ---
    # Symmetrical Semantic Overlap (Arithmetic Mean)
    # This represents the overall shared conceptual space between the two tools.
    unified_overlap_arithmetic = (coverage_B_by_A + coverage_A_by_B) / 2


    # -----------------------------------------------------------------
    # STEP D: RESULTS DISPLAY
    # -----------------------------------------------------------------
    print("\n" + "="*65)
    print("📊 SEMANTIC ANALYSIS RESULTS")
    print("="*65)

    print(f"\n[1] DIRECTIONAL CONTENT COVERAGE:")
    print(f"    👉 A covers B: {coverage_B_by_A:.4f}")
    print(f"    👉 B covers A: {coverage_A_by_B:.4f}")

    print(f"\n[2] UNIFIED OVERLAP SCORE (For publication):")
    print(f"    👉 SYMMETRICAL OVERLAP (Arithmetic Mean): {unified_overlap_arithmetic:.4f}")
    print("-" * 65)
    print("NOTE: Directional coverage indicates the extent to which one questionnaire")
    print("captures the specific topics addressed in the other.")

else:
    print("⚠️ Missing embeddings. Please check your API connection or Cell 2 configuration.")

--- 1. INITIALIZATION CHECK ---

--- STEP A: DATA DEFINITION ---
✅ Questionnaire A loaded: 24 items.
✅ Questionnaire B loaded: 21 items.

--- STEP B: EMBEDDING GENERATION ---
✅ Vectors generated successfully.

--- STEP C: CALCULATIONS ---

📊 SEMANTIC ANALYSIS RESULTS

[1] DIRECTIONAL CONTENT COVERAGE:
    👉 A covers B: 0.4502
    👉 B covers A: 0.4551

[2] UNIFIED OVERLAP SCORE (For publication):
    👉 SYMMETRICAL OVERLAP (Arithmetic Mean): 0.4527
-----------------------------------------------------------------
NOTE: Directional coverage indicates the extent to which one questionnaire
captures the specific topics addressed in the other.
